In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. Install Gradio if you haven't
!pip install gradio
!pip install torch torchvision face_recognition opencv-python numpy matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━

In [ ]:
# 2. Import libraries
import gradio as gr
from transformers import ViTForImageClassification, ViTImageProcessor
from transformers import BertForSequenceClassification, BertTokenizer
import torch
from PIL import Image
import timm
import torch.nn as nn # Import the 'nn' module from PyTorch
from torchvision import transforms, models
import numpy as np
import cv2


# 3. Load model and processor
image_model_path = '/content/drive/MyDrive/vit_deepfake_model'
original_model_path = '/content/drive/MyDrive/bert-original-caption-model'
generated_model_path = '/content/drive/MyDrive/bert-generated-caption-model'
video_model_path = '/content/drive/MyDrive/deepfake_project/video_model'

In [ ]:
import os
# 4. Load the actual models
try:
    # Check if the directory exists
    if not os.path.exists(image_model_path):
        raise ValueError(f"Image model directory not found: {image_model_path}")

    # Attempt to load the model and processor
    vit_model = ViTForImageClassification.from_pretrained(image_model_path).to('cpu')
    vit_processor = ViTImageProcessor.from_pretrained(image_model_path)

except Exception as e:
    print(f"Error loading ViT model: {e}")
    print(f"image_model_path: {image_model_path}")
    raise  # Re-raise the exception to see the full traceback

try:
    bert_original_model = BertForSequenceClassification.from_pretrained(original_model_path).to('cpu')
    bert_original_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
except Exception as e:
    print(f"Error loading original BERT model: {e}")
    print(f"original_model_path: {original_model_path}")
    raise

try:
    bert_generated_model = BertForSequenceClassification.from_pretrained(generated_model_path).to('cpu')
    bert_generated_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
except Exception as e:
    print(f"Error loading generated BERT model: {e}")
    print(f"generated_model_path: {generated_model_path}")
    raise

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
def create_video_model(model_name='efficientnet_b0', num_classes=2): # added create_video_model
    """Loads a pre-trained model."""
    if model_name.startswith('efficientnet'):
        model = timm.create_model(model_name, pretrained=False, num_classes=num_classes)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)
    elif model_name == 'resnet50':
        model = models.resnet50(pretrained=False)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
    elif model_name == 'resnext50_32x4d':
        model = timm.create_model(model_name, pretrained=False, num_classes=num_classes)
    # Assuming 'xception' is the intended model name based on the file path
    elif model_name == 'xception':
        model = timm.create_model(model_name, pretrained=False, num_classes=num_classes)
    else:
        raise ValueError(f"Model {model_name} not supported.")
    return model.to('cpu')  # Place the model on the CPU


video_model_path = '/content/drive/MyDrive/video model/best_model_xception.pth' # Assuming your model file is named 'video_model.pth' and is located inside the 'video model' directory.
video_model_name = 'xception' # Define video_model_name with the intended architecture
video_model = create_video_model(video_model_name)

# Load the state dict only if it contains the expected keys and shapes
try:
    state_dict = torch.load(video_model_path, map_location=torch.device('cpu'))
    video_model.load_state_dict(state_dict, strict=False)  # Set strict=False to ignore missing/unexpected keys
    print("Model weights loaded successfully!")
except RuntimeError as e:
    print(f"Error loading weights: {e}")
    print("Possible causes:")
    print("- Model architecture mismatch between the saved weights and the current model.")
    print("- Different input sizes or preprocessing used during training and inference.")
    print("Check if the model name and configuration match the saved weights.")


# Set models to evaluation mode
vit_model.eval()

!git clone https://github.com/abhijithjadhav/Deepfake_detection_using_deep_learning.git
%cd Deepfake_detection_using_deep_learning

/usr/local/lib/python3.11/dist-packages/timm/models/_factory.py:126: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Model weights loaded successfully!
Cloning into 'Deepfake_detection_using_deep_learning'...
remote: Enumerating objects: 410, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 410 (delta 65), reused 23 (delta 23), pack-reused 320 (from 3)
Receiving objects: 100% (410/410), 74.69 MiB | 15.75 MiB/s, done.
Resolving deltas: 100% (182/182), done.
/content/Deepfake_detection_using_deep_learning


In [ ]:
bert_original_model = BertForSequenceClassification.from_pretrained(original_model_path).to('cpu')
bert_original_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

bert_generated_model = BertForSequenceClassification.from_pretrained(generated_model_path).to('cpu')
bert_generated_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Set models to evaluation mode
vit_model.eval()
bert_original_model.eval()
bert_generated_model.eval()

# Image prediction (ViT)
def predict_image(image):
    inputs = vit_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = vit_model(**inputs)
    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    confidence, prediction = torch.max(probs, dim=1)

    label_map = {0: "Fake", 1: "Real"}
    label = label_map[prediction.item()]
    confidence_percent = confidence.item() * 100

    return f"Image Prediction: {label} ({confidence_percent:.2f}%)"

# Text prediction (BERT Original Captions)
def predict_text_original(text):
    inputs = bert_original_tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = bert_original_model(**inputs)
    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    confidence, prediction = torch.max(probs, dim=1)

    label_map = {0: "Fake News", 1: "Real News"}
    label = label_map[prediction.item()]
    confidence_percent = confidence.item() * 100

    return f"Original Caption Prediction: {label} ({confidence_percent:.2f}%)"

# Text prediction (BERT Generated Captions)
def predict_text_generated(text):
    inputs = bert_generated_tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = bert_generated_model(**inputs)
    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    confidence, prediction = torch.max(probs, dim=1)

    label_map = {0: "Fake News", 1: "Real News"}
    label = label_map[prediction.item()]
    confidence_percent = confidence.item() * 100

    return f"Generated Caption Prediction: {label} ({confidence_percent:.2f}%)"

def extract_frames(video_file, num_frames=16):
    """Extracts frames from a video file."""
    frames = []
    cap = cv2.VideoCapture(video_file)
    if not cap.isOpened():
        raise ValueError(f"Could not open video file: {video_file}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    for i in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            raise ValueError(f"Error reading frame {i} from {video_file}")
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()
    return frames

def preprocess_frames(frames):
    """Preprocesses the frames for the model."""
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    frames_tensor = [transform(frame) for frame in frames]
    frames_tensor = torch.stack(frames_tensor).to('cpu')  # Keep tensors on CPU
    return frames_tensor



def predict_video(video):
    """Predicts whether a video is real or fake.

    Args:
        video: A file path to the video.
    Returns:
        str:  The prediction.
    """

    frames = extract_frames(video)
    frames_tensor = preprocess_frames(frames)

    with torch.no_grad():
        outputs = video_model(frames_tensor)
        _, predicted = torch.max(outputs.data, 1)
        prediction = torch.round(torch.mean(predicted.float()))
    return "Fake" if prediction.item() else "Real"

def predict_uploaded_video(video_file):
    # video_file is the path to the uploaded video
    cap = cv2.VideoCapture(video_file)

    frames = []
    count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
        count += 1
        if count >= 150:  # limit to 150 frames if needed
            break
    cap.release()

# 6. Create the individual interfaces
image_interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="Upload an Image"),
    outputs=gr.Textbox(label="Prediction Result"),
    title="Deepfake Image Detector"
)

original_caption_interface = gr.Interface(
    fn=predict_text_original,
    inputs=gr.Textbox(lines=5, placeholder="Paste original caption text here..."),
    outputs=gr.Textbox(label="Prediction Result"),
    title="Fake News Detection (Original Captions)"
)

generated_caption_interface = gr.Interface(
    fn=predict_text_generated,
    inputs=gr.Textbox(lines=5, placeholder="Paste generated caption text here..."),
    outputs=gr.Textbox(label="Prediction Result"),
    title="Fake News Detection (Generated Captions)"
)

video_interface = gr.Interface(
    fn=predict_uploaded_video,
    inputs=gr.Video(),
    outputs=gr.Textbox(label="Prediction Result"),
    title="Deepfake Detection",
    description="Upload a video to classify it as real or fake."
)

# 7. Group into tabs
gr.TabbedInterface(
    [image_interface, original_caption_interface, generated_caption_interface, video_interface], #added video_interface
    tab_names=["Detect Deepfake Image", "Detect Fake News (Original Captions)", "Detect Fake News (Generated Captions)", "Detect Deepfake Video"] #added tab
).launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cb757c0b77f2ce5cb4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Deepfake Detection - Gradio Interface using Pre-trained ResNeXt + LSTM Model

# Install required libraries
!pip install torch torchvision face_recognition opencv-python numpy matplotlib gradio

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data.dataset import Dataset
import numpy as np
import cv2
import dlib
import face_recognition
import gradio as gr

# --------------------------
# Model Definition
# --------------------------
class Model(nn.Module):
    def __init__(self, num_classes, latent_dim=2048, lstm_layers=1, hidden_dim=2048, bidirectional=False):
        super(Model, self).__init__()
        model = models.resnext50_32x4d(pretrained=True)
        self.model = nn.Sequential(*list(model.children())[:-2])
        self.lstm = nn.LSTM(latent_dim, hidden_dim, lstm_layers, bidirectional)
        self.relu = nn.LeakyReLU()
        self.dp = nn.Dropout(0.4)
        self.linear1 = nn.Linear(2048, num_classes)
        self.avgpool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        batch_size, seq_length, c, h, w = x.shape
        x = x.view(batch_size * seq_length, c, h, w)
        fmap = self.model(x)
        x = self.avgpool(fmap)
        x = x.view(batch_size, seq_length, 2048)
        x_lstm, _ = self.lstm(x, None)
        return fmap, self.dp(self.linear1(x_lstm[:, -1, :]))

# --------------------------
# Load Pre-trained Model
# --------------------------
model = Model(2).cuda()
model.load_state_dict(torch.load('/content/drive/My Drive/Models/model_87_acc_20_frames_final_data.pt'))
model.eval()

# --------------------------
# Dataset and Preprocessing
# --------------------------
im_size = 112
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((im_size, im_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

class validation_dataset(Dataset):
    def __init__(self, video_names, sequence_length=20, transform=None):
        self.video_names = video_names
        self.transform = transform
        self.count = sequence_length

    def __len__(self):
        return len(self.video_names)

    def __getitem__(self, idx):
        video_path = self.video_names[idx]
        frames = []
        for i, frame in enumerate(self.frame_extract(video_path)):
            faces = face_recognition.face_locations(frame)
            try:
                top, right, bottom, left = faces[0]
                frame = frame[top:bottom, left:right, :]
            except:
                pass  # Use full frame if no face detected
            frames.append(self.transform(frame))
            if len(frames) == self.count:
                break
        frames = torch.stack(frames)
        frames = frames[:self.count]
        return frames.unsqueeze(0)

    def frame_extract(self, path):
        vidObj = cv2.VideoCapture(path)
        success = 1
        while success:
            success, image = vidObj.read()
            if success:
                yield image

# --------------------------
# Prediction Function
# --------------------------
sm = nn.Softmax(dim=1)

def predict(model, img):
    fmap, logits = model(img.cuda())
    logits = sm(logits)
    _, prediction = torch.max(logits, 1)
    confidence = logits[:, int(prediction.item())].item() * 100
    label = "REAL" if prediction.item() == 1 else "FAKE"
    return label, confidence

# --------------------------
# Gradio Integration
# --------------------------
def predict_uploaded_video(video_file):
    dataset = validation_dataset([video_file], sequence_length=20, transform=train_transforms)
    video_tensor = dataset[0]  # Only one video at a time
    label, confidence = predict(model, video_tensor)
    return f"Prediction: {label} (Confidence: {confidence:.2f}%)"

interface = gr.Interface(
    fn=predict_uploaded_video,
    inputs=gr.Video(),
    outputs=gr.Textbox(label="Prediction Result"),
    title="Deepfake Detection Video Demo",
    description="Upload a video to classify it as real or fake."
)

interface.launch()

RuntimeError: Error while calling cudaGetDevice(&the_device_id) in file /tmp/.tmp3TzdkI/sdists-v9/pypi/dlib/19.24.6/A6Becce7QDu-gtWWWa8DY/src/dlib/cuda/gpu_data.cpp:204. code: 35, reason: CUDA driver version is insufficient for CUDA runtime version